## Step 1. Imports e configuração

In [0]:
from pyspark.sql import functions as F

from notebooks._shared.configuration import AppConfig
from notebooks._shared.contracts import SILVER_CONTRACTS

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("silver_schema", "")

In [0]:
config = AppConfig(
    catalog=dbutils.widgets.get("catalog"),
    silver_schema=dbutils.widgets.get("silver_schema"),
)

## Step 2. Leitura das entidades Silver

In [0]:
silver_dataframes = {
    contract.name: spark.table(f"{config.silver_namespace}.{contract.name}")
    for contract in SILVER_CONTRACTS
}

for entity, dataframe in silver_dataframes.items():
    print(f"{entity:<25}: {dataframe.count()} linhas")

## Step 3. Validação contratual

Confronta cada entidade materializada com seu contrato versionado, verificando estrutura, campos obrigatórios e unicidade da chave

In [0]:
contract_results = []

for contract in SILVER_CONTRACTS:
    dataframe = silver_dataframes[contract.name]

    expected_fields = [
        (field.name, field.dataType.simpleString()) for field in contract.schema.fields
    ]
    actual_fields = [
        (field.name, field.dataType.simpleString()) for field in dataframe.schema.fields
    ]

    schema_valid = actual_fields == expected_fields

    required_columns = [
        field.name for field in contract.schema.fields if not field.nullable
    ]

    required_nulls = (
        dataframe.select(
            *[
                F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
                for column in required_columns
            ]
        )
        .first()
        .asDict()
    )

    required_valid = all(count == 0 for count in required_nulls.values())

    duplicate_keys = (
        dataframe.groupBy(*contract.key).count().filter(F.col("count") > 1).count()
    )

    key_valid = duplicate_keys == 0

    contract_results.append(
        (
            contract.name,
            schema_valid,
            required_valid,
            duplicate_keys,
            key_valid,
        )
    )

    print(
        f"{contract.name:<25}: "
        f" schema={schema_valid}, "
        f" obrigatórios={required_valid}, "
        f" duplicidades_chave={duplicate_keys}"
    )

## Step 4. Integridade referencial

Verifica se os filmes referenciados pelas entidades normalizadas existem na entidade principal `movie`

In [0]:
movie_ids_df = silver_dataframes["movie"].select("movie_id")

dependent_entities = [
    contract.name for contract in SILVER_CONTRACTS if contract.name != "movie"
]

referential_results = {}

for entity in dependent_entities:
    orphan_count = (
        silver_dataframes[entity]
        .select("movie_id")
        .distinct()
        .join(movie_ids_df, on="movie_id", how="left_anti")
        .count()
    )

    referential_results[entity] = orphan_count

    print(f"{entity:<25}: movie_id órfãos={orphan_count}")

## Step 5. Resultado consolidado

Consolida as verificações e falha explicitamente caso alguma entidade Silver viole o contrato ou a integridade referencial esperada

In [0]:
referential_results

In [0]:
contract_violations = [
    entity
    for entity, schema_valid, required_valid, duplicate_keys, key_valid in contract_results
    if not schema_valid or not required_valid or not key_valid
]

referential_violations = [
    entity for entity, orphan_count in referential_results.items() if orphan_count > 0
]

if contract_violations or referential_violations:
    raise RuntimeError(
        "Validação Silver falhou. "
        f"Violações contratuais={contract_violations}; "
        f"violações referenciais={referential_violations}."
    )

print("Silver quality validation completed successfully")
print(f"Silver Namespace: {config.silver_namespace}")
print(f"Contracts Validated: {len(SILVER_CONTRACTS)}")